In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# Docking a single ligand 

This notebook shows you how to dock a single ligand to a protein. 

## Setup

First, we'll import the necessary Deep Origin drug discovery modules.


In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Pocket,
    Protein,
    Protonation,
    Docking,
    Ligand,
    LigandSet,
)

from deeporigin.platform import DeepOriginClient

You don't have to explictitly initialize a client, but you can if you want:

In [ ]:
client = DeepOriginClient()
client

## Load protein and register on the platform

We use the the same BRD protein as in our other notebooks, and use the `sync` method to upload the PDB file and register it with the data platform:


In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.remove_water()
protein.sync()
protein.id

## Load ligand

We load a ligand from a SDF file on disk. You can also import a ligand from a SMILES string, etc. Once again, we use the `sync` method to upload the file (if any) and register with the data platform.

In [ ]:
ligand = Ligand.from_sdf(BRD_DATA_DIR/"brd-2.sdf")
ligand.sync()
ligand

In [ ]:
ligand.id

## Protonate Ligand

We use the protonation tool to protonate the ligand and use the most probable species at that pH to dock. Note that the Protonator tool can modify our ligand. 

In [ ]:
protonator = Protonation(ligand=ligand)
protonator.run()

## Work with a pocket

Here, we will use a previously identified novel pocket using the PocketFinder tool.

In [ ]:
pockets = Pocket.from_result(protein_id=protein.id)
pocket = pockets[0]
pocket

## Show pocket

Here, we view the pocket in the protein:

In [ ]:
protein.show(pockets=[pocket])

## Show docking box

We can also view the docking box that is constructed from this pokcet:



In [ ]:
docking = Docking(protein=protein, pocket=pockets[0], ligand=ligand)
docking.show_box()

## Estimate cost

Before running any tool, we can estimate the cost by passing `quote=True` as follows:



In [ ]:

docking.run(quote=True)
docking.estimate

## Dock ligand

We can now run the docking tool. Because we're docking a single ligand, we get back poses immediately. `Docking.run` (like all `run` methods) is a blocking synchronouse operation. 


In [ ]:
poses = docking.run()

We can view all poses in a dataframe:

In [ ]:
poses.to_dataframe()

## Show poses

To visualize the poses, we download them and show them in the protein:

In [ ]:
poses.download()
protein.show(poses=poses)
